In [0]:
BASE_PATH = "/Volumes/workspace/default/e-commerce_recommendation/"

In [0]:
orders_df = spark.read.csv(BASE_PATH + "orders.csv",
                           header=True,
                           inferSchema=True)

products_df = spark.read.csv(BASE_PATH + "products.csv",
                             header=True,
                             inferSchema=True)

prior_df = spark.read.csv(BASE_PATH + "order_products__prior.csv",
                          header=True,
                          inferSchema=True)

train_df = spark.read.csv(BASE_PATH + "order_products__train.csv",
                          header=True,
                          inferSchema=True)

aisles_df = spark.read.csv(BASE_PATH + "aisles.csv",
                           header=True,
                           inferSchema=True)

departments_df = spark.read.csv(BASE_PATH + "departments.csv",
                                header=True,
                                inferSchema=True)

In [0]:
from pyspark.sql.functions import col

products_df = (
    products_df
    .withColumn("aisle_id", col("aisle_id").cast("int"))
    .withColumn("department_id", col("department_id").cast("int"))
)

products_df.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- aisle_id: integer (nullable = true)
 |-- department_id: integer (nullable = true)



In [0]:
products_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "true")
    .option("escape", '"')
    .option("quote", '"')
    .csv(BASE_PATH + "products.csv")
)

In [0]:
products_df.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- aisle_id: integer (nullable = true)
 |-- department_id: integer (nullable = true)



In [0]:
products_df.filter(
    products_df.product_id == 6816
).show(truncate=False)

+----------+--------------------------------------+--------+-------------+
|product_id|product_name                          |aisle_id|department_id|
+----------+--------------------------------------+--------+-------------+
|6816      |Scotch Kids 5\" Scissors, Blunted, Red|87      |17           |
+----------+--------------------------------------+--------+-------------+



# Save Bronze Tables

In [0]:
orders_df.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("bronze_orders")

products_df.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("bronze_products")

prior_df.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("bronze_prior")

train_df.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("bronze_train")

aisles_df.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("bronze_aisles")

departments_df.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("bronze_departments")

In [0]:
spark.sql("SHOW TABLES").show()

+--------+------------------+-----------+
|database|         tableName|isTemporary|
+--------+------------------+-----------+
| default|     bronze_aisles|      false|
| default|bronze_departments|      false|
| default|     bronze_orders|      false|
| default|      bronze_prior|      false|
| default|   bronze_products|      false|
| default|      bronze_train|      false|
| default|      clean_orders|      false|
+--------+------------------+-----------+



In [0]:
raw = spark.read.text(BASE_PATH + "products.csv")

display(raw.limit(20))

value
"product_id,product_name,aisle_id,department_id"
"1,Chocolate Sandwich Cookies,61,19"
"2,All-Seasons Salt,104,13"
"3,Robust Golden Unsweetened Oolong Tea,94,7"
"4,Smart Ones Classic Favorites Mini Rigatoni With Vodka Cream Sauce,38,1"
"5,Green Chile Anytime Sauce,5,13"
"6,Dry Nose Oil,11,11"
"7,Pure Coconut Water With Orange,98,7"
"8,Cut Russet Potatoes Steam N' Mash,116,1"
"9,Light Strawberry Blueberry Yogurt,120,16"


## # Silver Tables

In [0]:
silver_df = (
    prior_df
    .join(orders_df, on='order_id', how='inner')
)

In [0]:
display(silver_df.limit(5))

order_id,product_id,add_to_cart_order,reordered,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
46,36884,1,0,10951,prior,3,0,18,18.0
46,48183,2,0,10951,prior,3,0,18,18.0
46,26172,3,0,10951,prior,3,0,18,18.0
46,4778,4,1,10951,prior,3,0,18,18.0
46,34519,5,1,10951,prior,3,0,18,18.0


In [0]:
silver_df = (
    silver_df
    .join(products_df, on='product_id', how='inner')
)

In [0]:
silver_df = (
    silver_df
    .join(aisles_df, on='aisle_id', how='inner')
)

In [0]:
silver_df = (
    silver_df
    .join(departments_df, on='department_id', how='inner')
)

In [0]:
display(silver_df.limit(5))

department_id,aisle_id,product_id,order_id,add_to_cart_order,reordered,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_name,aisle,department
3,112,8467,475,1,1,190994,prior,3,0,0,6.0,Cinnamon Raisin Bread,bread,bakery
4,24,24852,475,2,1,190994,prior,3,0,0,6.0,Banana,fresh fruits,produce
1,52,9515,475,3,0,190994,prior,3,0,0,6.0,Natural Classic Pork Breakfast Sausage,frozen breakfast,frozen
4,24,28204,475,4,0,190994,prior,3,0,0,6.0,Organic Fuji Apple,fresh fruits,produce
4,123,48205,475,5,0,190994,prior,3,0,0,6.0,Spinach,packaged vegetables fruits,produce


In [0]:
silver_df.printSchema()

root
 |-- department_id: integer (nullable = true)
 |-- aisle_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- add_to_cart_order: integer (nullable = true)
 |-- reordered: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- eval_set: string (nullable = true)
 |-- order_number: integer (nullable = true)
 |-- order_dow: integer (nullable = true)
 |-- order_hour_of_day: integer (nullable = true)
 |-- days_since_prior_order: double (nullable = true)
 |-- product_name: string (nullable = true)
 |-- aisle: string (nullable = true)
 |-- department: string (nullable = true)



In [0]:
products_df.filter(
    (~col("aisle_id").rlike("^[0-9]+$")) |
    (~col("department_id").rlike("^[0-9]+$"))
).show(20, truncate=False)

+----------+------------+--------+-------------+
|product_id|product_name|aisle_id|department_id|
+----------+------------+--------+-------------+
+----------+------------+--------+-------------+



In [0]:
silver_df.limit(5).show(truncate=False)

+-------------+--------+----------+--------+-----------------+---------+-------+--------+------------+---------+-----------------+----------------------+--------------------------------------+--------------------------+----------+
|department_id|aisle_id|product_id|order_id|add_to_cart_order|reordered|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|product_name                          |aisle                     |department|
+-------------+--------+----------+--------+-----------------+---------+-------+--------+------------+---------+-----------------+----------------------+--------------------------------------+--------------------------+----------+
|3            |112     |8467      |475     |1                |1        |190994 |prior   |3           |0        |0                |6.0                   |Cinnamon Raisin Bread                 |bread                     |bakery    |
|4            |24      |24852     |475     |2                |1        |1909

In [0]:
silver_df.count()

32434489

In [0]:
silver_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_customer_products")

## GOLD LAYER

In [0]:
from pyspark.sql.functions import count

gold_df = (
    silver_df
    .groupBy("user_id","product_id")
    .agg(
        count("*").alias("rating")
    )
)

In [0]:
test_df = silver_df.limit(1000)

gold_test = (
    test_df
    .groupBy("user_id","product_id")
    .agg(count("*").alias("rating"))
)

display(gold_test)

user_id,product_id,rating
190994,8467,1
190994,24852,1
190994,9515,1
190994,28204,1
190994,48205,1
34791,5120,1
34791,37215,1
34791,33395,1
34791,16953,1
34791,6087,1


In [0]:
gold_df.write\
.format("delta")\
.mode("overwrite")\
.saveAsTable("gold_interaction_matrix")